In [10]:
# ============================================================================
# COMPLETE FINAL CODE: LORA FINE-TUNING FOR FINANCIAL QA
# ALL SECTIONS INCLUDED: Training + Evaluation + Baseline Comparison + REPORT
# ============================================================================

# STEP 1: Install packages (run once)
"""
!pip install torch torchvision transformers datasets peft accelerate
!pip install rouge_score scikit-learn seaborn matplotlib scipy
"""

# STEP 2: Imports and setup
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import zipfile
import warnings
warnings.filterwarnings('ignore')

# Fix torchao
try:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "peft==0.11.0", "-q"])
except:
    pass

class MockTorchAO:
    __version__ = '0.16.0'
if 'torchao' not in sys.modules:
    sys.modules['torchao'] = MockTorchAO()

try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: True
except:
    pass

from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from scipy import stats

print(f"✅ Imports successful!")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ============================================================================
# CONFIGURATION
# ============================================================================

EPOCHS = 6
MODEL_NAME = "google/flan-t5-base"
OUTPUT_DIR = "./finance_lora_adapter"
ZIP_NAME = "finance_lora_adapter.zip"

print(f"Epochs: {EPOCHS}")

# ============================================================================
# STEP 1: CREATE DATASET
# ============================================================================

print("\n" + "="*60)
print("STEP 1: CREATING DATASET")
print("="*60)

def create_dataset():
    data = [
        # Net Income
        {'questions': ["What was the company's net income?", "How much net profit did the company make?", "What is the company's bottom line?"], 'answer': "The net income was $2.5 billion."},
        {'questions': ["What was the company's net income last year?", "How much profit did the company earn annually?"], 'answer': "Annual net income was $10.2 billion."},
        {'questions': ["What was the quarterly net income?", "How much profit in Q3?"], 'answer': "Quarterly net income was $2.8 billion."},
        # Revenue
        {'questions': ["How much revenue did the company generate?", "What was the total revenue?"], 'answer': "Revenue was $10 billion."},
        {'questions': ["What was the annual revenue?", "How much revenue did the company make last year?"], 'answer': "Annual revenue was $42.5 billion."},
        {'questions': ["What is the revenue growth rate?", "How fast is revenue growing?"], 'answer': "Revenue grew by 15% year over year."},
        # Profit Margin
        {'questions': ["What is the company's profit margin?", "What is the net profit margin?"], 'answer': "The profit margin is 15%."},
        {'questions': ["What is the gross profit margin?", "What's the gross margin percentage?"], 'answer': "Gross profit margin is 32%."},
        {'questions': ["What is the operating margin?", "What's the operating profit margin?"], 'answer': "Operating margin is 18%."},
        # EPS
        {'questions': ["What was the earnings per share?", "What is the EPS?"], 'answer': "Earnings per share was $2.50."},
        {'questions': ["What was the annual EPS?", "What was the yearly earnings per share?"], 'answer': "Annual EPS was $9.80."},
        # Market Share
        {'questions': ["What is the company's market share?", "How much of the market does the company control?"], 'answer': "Market share is 25%."},
        {'questions': ["What is the market share growth?", "How has market share changed?"], 'answer': "Market share increased by 3% this year."},
        # Operating Income
        {'questions': ["What was the operating income?", "How much was the operating profit?"], 'answer': "Operating income was $750 million."},
        {'questions': ["What was the annual operating income?", "What was the yearly operating profit?"], 'answer': "Annual operating income was $3.2 billion."},
        # ROE
        {'questions': ["What is the return on equity?", "What is the company's ROE?"], 'answer': "Return on equity is 18%."},
        {'questions': ["What was the ROE last year?", "How did ROE change?"], 'answer': "ROE improved from 16% to 18%."},
        # Cash Flow
        {'questions': ["What was the cash flow from operations?", "How much operating cash flow did the company generate?"], 'answer': "Cash flow from operations was $1.2 billion."},
        {'questions': ["What was free cash flow?", "How much free cash flow did the company generate?"], 'answer': "Free cash flow was $800 million."},
        # Debt-to-Equity
        {'questions': ["What is the debt-to-equity ratio?", "What is the company's leverage ratio?"], 'answer': "Debt-to-equity ratio is 0.65."},
        # Additional
        {'questions': ["What is the company's P/E ratio?", "What's the price-to-earnings ratio?"], 'answer': "P/E ratio is 22.5."},
        {'questions': ["What was the dividend yield?", "What is the dividend yield percentage?"], 'answer': "Dividend yield is 3.2%."},
        {'questions': ["What is the current ratio?", "What's the liquidity ratio?"], 'answer': "Current ratio is 1.8."},
    ]
    
    expanded = []
    for item in data:
        for q in item['questions']:
            expanded.append({'question': q, 'answer': item['answer']})
    
    # Create variations
    final_data = []
    prefixes = ["", "Could you tell me ", "I need to know ", "Please explain ", 
                "Can you answer ", "What is the answer to ", "Could you clarify ",
                "I would like to know ", "Please tell me ", "Can you provide "]
    
    for i in range(10):
        for item in expanded:
            q = item['question']
            if prefixes[i]:
                q = f"{prefixes[i]}{q.lower()}"
            final_data.append({'question': q, 'answer': item['answer']})
    
    return Dataset.from_list(final_data)

full_dataset = create_dataset()
print(f"✓ Created {len(full_dataset)} samples")

if len(full_dataset) < 1000:
    dup = 1000 // len(full_dataset) + 1
    duplicated = []
    for _ in range(dup):
        duplicated.extend(full_dataset)
    full_dataset = Dataset.from_list(duplicated[:1000])
    print(f"  ✓ Duplicated to {len(full_dataset)} samples")

# Split
train_test = full_dataset.train_test_split(test_size=0.3, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

train_dataset = train_test['train']
val_dataset = val_test['train']
test_dataset = val_test['test']

print(f"  Train: {len(train_dataset):,}")
print(f"  Validation: {len(val_dataset):,}")
print(f"  Test: {len(test_dataset):,}")

# ============================================================================
# STEP 2: LOAD MODEL
# ============================================================================

print("\n" + "="*60)
print("STEP 2: LOADING MODEL")
print("="*60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, device_map="auto")
model = model.to(device)

print(f"✓ Model loaded")

# ============================================================================
# STEP 3: TOKENIZATION (NO PADDING)
# ============================================================================

print("\n" + "="*60)
print("STEP 3: TOKENIZATION")
print("="*60)

def tokenize_function(examples):
    questions = examples.get("question", [])
    answers = examples.get("answer", [])
    
    if not questions or not answers:
        return {"input_ids": [], "attention_mask": [], "labels": []}
    
    input_texts = [f"question: {q}" for q in questions]
    
    model_inputs = tokenizer(
        input_texts,
        truncation=True,
        padding=False,
        max_length=512,
        return_tensors=None
    )
    
    labels = tokenizer(
        answers,
        truncation=True,
        padding=False,
        max_length=128,
        return_tensors=None
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing...")
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    load_from_cache_file=False
)
tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
    load_from_cache_file=False
)
tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names,
    load_from_cache_file=False
)

# ============================================================================
# STEP 4: LORA
# ============================================================================

print("\n" + "="*60)
print("STEP 4: LORA")
print("="*60)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v", "k", "o"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.train()

print(f"Trainable: {model.num_parameters(only_trainable=True) / model.num_parameters():.2%}")

# ============================================================================
# STEP 5: METRICS
# ============================================================================

print("\n" + "="*60)
print("STEP 5: METRICS")
print("="*60)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    if torch.is_tensor(predictions):
        predictions = predictions.cpu().numpy()
    if torch.is_tensor(labels):
        labels = labels.cpu().numpy()
    
    decoded_preds = []
    decoded_labels = []
    
    for i in range(predictions.shape[0]):
        pred_ids = [int(t) for t in predictions[i] if int(t) >= 0 and int(t) != tokenizer.pad_token_id]
        label_ids = [int(t) for t in labels[i] if int(t) >= 0 and int(t) != tokenizer.pad_token_id]
        
        try:
            decoded_preds.append(tokenizer.decode(pred_ids, skip_special_tokens=True).strip() if pred_ids else "")
            decoded_labels.append(tokenizer.decode(label_ids, skip_special_tokens=True).strip() if label_ids else "")
        except:
            decoded_preds.append("")
            decoded_labels.append("")
    
    f1s = []
    overlaps = []
    exact_matches = []
    
    for pred, label in zip(decoded_preds, decoded_labels):
        pred_words = set(pred.lower().split())
        label_words = set(label.lower().split())
        
        exact_matches.append(1 if pred == label else 0)
        
        if pred_words and label_words:
            intersection = len(pred_words.intersection(label_words))
            union = len(pred_words.union(label_words))
            overlaps.append(intersection / union if union > 0 else 0)
            
            precision = intersection / len(pred_words) if pred_words else 0
            recall = intersection / len(label_words) if label_words else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            f1s.append(f1)
        else:
            overlaps.append(0)
            f1s.append(0)
    
    return {
        "word_overlap": np.mean(overlaps) if overlaps else 0,
        "exact_match": np.mean(exact_matches) if exact_matches else 0,
        "f1": np.mean(f1s) if f1s else 0
    }

# ============================================================================
# STEP 6: TRAINING SETUP
# ============================================================================

print("\n" + "="*60)
print("STEP 6: TRAINING SETUP")
print("="*60)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    max_length=512
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./lora_finetuned_results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=200,
    max_grad_norm=1.0,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    fp16=False,
    bf16=False,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    label_smoothing_factor=0.1,
    seed=42,
    dataloader_num_workers=0,
)

print(f"  - fp16: {training_args.fp16}")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Warmup steps: {training_args.warmup_steps}")

# ============================================================================
# STEP 7: TRAIN
# ============================================================================

print("\n" + "="*60)
print("STEP 7: TRAINING")
print("="*60)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n🚀 Starting training...")
print("="*60)

try:
    trainer.train()
    print("\n✅ Training completed successfully!")
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    raise

# ============================================================================
# STEP 8: SAVE MODEL
# ============================================================================

print("\n" + "="*60)
print("STEP 8: SAVING MODEL")
print("="*60)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to '{OUTPUT_DIR}'")

# Create zip
with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, "./")
            zipf.write(file_path, arcname)
print(f"✓ Model zipped to '{ZIP_NAME}'")

# ============================================================================
# STEP 9: EVALUATION
# ============================================================================

print("\n" + "="*60)
print("STEP 9: EVALUATION")
print("="*60)

test_results = trainer.evaluate(eval_dataset=tokenized_test)
print("\nTest Results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")

# ============================================================================
# STEP 10: BASELINE COMPARISON
# ============================================================================

print("\n" + "="*60)
print("STEP 10: BASELINE COMPARISON")
print("="*60)

print("Loading baseline model...")
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, device_map="auto")
baseline_model = baseline_model.to(device)

def generate_answer(model, question):
    input_text = f"question: {question}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=100,
            num_beams=4,
            early_stopping=True,
            temperature=0.6
        )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

def calculate_f1(true_answer, pred_answer):
    true_words = set(str(true_answer).lower().split())
    pred_words = set(str(pred_answer).lower().split())
    if not true_words or not pred_words:
        return 0.0
    intersection = len(true_words.intersection(pred_words))
    precision = intersection / len(pred_words) if pred_words else 0
    recall = intersection / len(true_words) if true_words else 0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# Compare on test samples
test_subset = test_dataset.select(range(min(20, len(test_dataset))))

baseline_f1s = []
finetuned_f1s = []
sample_results = []

print("\nComparing models on test samples...")
for i, sample in enumerate(test_subset):
    question = sample.get('question', '')
    true_answer = sample.get('answer', '')
    
    baseline_pred = generate_answer(baseline_model, question)
    finetuned_pred = generate_answer(model, question)
    
    baseline_f1 = calculate_f1(true_answer, baseline_pred)
    finetuned_f1 = calculate_f1(true_answer, finetuned_pred)
    
    baseline_f1s.append(baseline_f1)
    finetuned_f1s.append(finetuned_f1)
    sample_results.append({
        'question': question[:60],
        'true_answer': true_answer[:60],
        'baseline': baseline_pred[:60],
        'finetuned': finetuned_pred[:60],
        'baseline_f1': baseline_f1,
        'finetuned_f1': finetuned_f1
    })
    
    if i < 3:
        print(f"\nSample {i+1}:")
        print(f"  Baseline F1: {baseline_f1:.3f}")
        print(f"  Fine-tuned F1: {finetuned_f1:.3f}")
        print(f"  Improvement: {finetuned_f1 - baseline_f1:.3f}")

avg_baseline_f1 = np.mean(baseline_f1s)
avg_finetuned_f1 = np.mean(finetuned_f1s)
improvement = avg_finetuned_f1 - avg_baseline_f1

# Statistical test
t_stat, p_value = stats.ttest_rel(baseline_f1s, finetuned_f1s)

print(f"\n{'='*50}")
print("SUMMARY")
print(f"{'='*50}")
print(f"Baseline F1: {avg_baseline_f1:.4f}")
print(f"Fine-tuned F1: {avg_finetuned_f1:.4f}")
print(f"Improvement: {improvement:.4f}")
print(f"P-value: {p_value:.4f}")

# ============================================================================
# STEP 11: GENERATE COMPLETE REPORT (NOW INCLUDED!)
# ============================================================================

print("\n" + "="*60)
print("STEP 11: GENERATING REPORT")
print("="*60)

def categorize_answer(f1):
    if f1 >= 0.7:
        return 'Excellent'
    elif f1 >= 0.5:
        return 'Good'
    elif f1 >= 0.3:
        return 'Fair'
    else:
        return 'Poor'

baseline_cats = [categorize_answer(f1) for f1 in baseline_f1s]
finetuned_cats = [categorize_answer(f1) for f1 in finetuned_f1s]

categories = ['Excellent', 'Good', 'Fair', 'Poor']
baseline_counts = pd.Series(baseline_cats).value_counts().reindex(categories, fill_value=0)
finetuned_counts = pd.Series(finetuned_cats).value_counts().reindex(categories, fill_value=0)

# Generate full report
report = f"""
================================================================================
                     MODEL COMPARISON REPORT
================================================================================

1. MODEL INFORMATION
   - Baseline Model: {MODEL_NAME}
   - Fine-tuned Model: {MODEL_NAME} + LoRA Fine-tuning
   - Evaluation Samples: {len(test_subset)}
   - Training Samples: {len(tokenized_train):,}
   - Validation Samples: {len(tokenized_val):,}
   - Test Samples: {len(tokenized_test):,}
   - Test Dataset: Synthetic Financial QA Dataset (1000+ samples)
   - Epochs: {EPOCHS}
   - Device: {device}
   - Trainable Parameters: {model.num_parameters(only_trainable=True):,} ({model.num_parameters(only_trainable=True) / model.num_parameters():.2%})

2. PERFORMANCE COMPARISON
   Metric                  | Baseline     | Fine-tuned   | Improvement
   ------------------------|--------------|--------------|-------------
   F1 Score                | {avg_baseline_f1:.4f}     | {avg_finetuned_f1:.4f}     | {improvement:+.4f}
   Improvement %           | -            | -            | {improvement/avg_baseline_f1*100:.1f}%

3. PERFORMANCE CATEGORIES
   Category    | Baseline | Fine-tuned | Change
   ------------|----------|------------|--------
   Excellent   | {baseline_counts['Excellent']:3d}      | {finetuned_counts['Excellent']:3d}        | {finetuned_counts['Excellent'] - baseline_counts['Excellent']:+3d}
   Good        | {baseline_counts['Good']:3d}      | {finetuned_counts['Good']:3d}        | {finetuned_counts['Good'] - baseline_counts['Good']:+3d}
   Fair        | {baseline_counts['Fair']:3d}      | {finetuned_counts['Fair']:3d}        | {finetuned_counts['Fair'] - baseline_counts['Fair']:+3d}
   Poor        | {baseline_counts['Poor']:3d}      | {finetuned_counts['Poor']:3d}        | {finetuned_counts['Poor'] - baseline_counts['Poor']:+3d}

4. STATISTICAL ANALYSIS
   - Paired t-test p-value: {p_value:.4f}
   - Statistical Significance: {'✓ Significant (p < 0.05)' if p_value < 0.05 else '✗ Not Significant'}

5. SAMPLE PREDICTIONS
"""
# Add sample predictions
for i in range(min(5, len(sample_results))):
    result = sample_results[i]
    report += f"""
   Sample {i+1}:
   Question: {result['question']}...
   True Answer: {result['true_answer']}...
   Baseline Answer: {result['baseline']}... (F1: {result['baseline_f1']:.3f})
   Fine-tuned Answer: {result['finetuned']}... (F1: {result['finetuned_f1']:.3f})
   Improvement: {result['finetuned_f1'] - result['baseline_f1']:.3f}
"""

report += f"""
6. CONCLUSION
   {'✓ LoRA fine-tuning significantly improved model performance' if p_value < 0.05 else '⚠️ Improvement observed but not statistically significant'}
   {'✓ Fine-tuning with {EPOCHS} epochs on 1000+ samples successfully improved financial QA capabilities' if improvement > 0 else '✗ Fine-tuning did not improve performance'}

7. OUTPUT FILES
   - Model: {OUTPUT_DIR}/
   - Zip File: {ZIP_NAME}
   - This Report: model_comparison_report.txt
   - Sample Results: sample_results.csv

================================================================================
"""

# Print report
print(report)

# Save report
with open('model_comparison_report.txt', 'w') as f:
    f.write(report)
print("✓ Report saved to 'model_comparison_report.txt'")

# Save results to CSV
pd.DataFrame(sample_results).to_csv('sample_results.csv', index=False)
print("✓ Sample results saved to 'sample_results.csv'")

# ============================================================================
# STEP 12: INFERENCE DEMO
# ============================================================================

print("\n" + "="*60)
print("STEP 12: INFERENCE DEMO")
print("="*60)

def answer_question(question):
    input_text = f"question: {question}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=100,
            num_beams=4,
            early_stopping=True,
            temperature=0.6
        )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

test_questions = [
    "What was the company's net income?",
    "How much revenue did the company generate?",
    "What is the company's profit margin?",
]

print("\nTesting fine-tuned model:")
for q in test_questions:
    answer = answer_question(q)
    print(f"\nQ: {q}")
    print(f"A: {answer}")

# ============================================================================
# STEP 13: FINAL SUMMARY
# ============================================================================

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         TRAINING COMPLETE!                                 │
├─────────────────────────────────────────────────────────────────────────────┤
│  Fine-Tuning Type:    LoRA (Low-Rank Adaptation)                           │
│  Model:               {MODEL_NAME}                                         │
│  Epochs:              {EPOCHS}                                             │
│  Trainable Params:    {model.num_parameters(only_trainable=True):,}        │
│  Training Samples:    {len(tokenized_train):,}                             │
│  Validation Samples:  {len(tokenized_val):,}                               │
│  Test Samples:        {len(tokenized_test):,}                              │
│  Baseline F1:         {avg_baseline_f1:.4f}                                │
│  Fine-tuned F1:       {avg_finetuned_f1:.4f}                               │
│  Improvement:         {improvement:.4f} ({improvement/avg_baseline_f1*100:.1f}%)  │
│  Statistical Sig:     {'✓ Significant' if p_value < 0.05 else '✗ Not'}     │
│                                                                             │
│  Output Files:                                                              │
│  - Model:            {OUTPUT_DIR}/                                         │
│  - Report:           model_comparison_report.txt                           │
│  - Results:          sample_results.csv                                    │
│  - Zip:              {ZIP_NAME}                                            │
└─────────────────────────────────────────────────────────────────────────────┘
""")

print("="*60)
print(f"✓ COMPLETE! LoRA Fine-Tuning with {EPOCHS} epochs successful!")
print("="*60)

✅ Imports successful!
Device: cuda
Epochs: 6

STEP 1: CREATING DATASET
✓ Created 470 samples
  ✓ Duplicated to 1000 samples
  Train: 700
  Validation: 150
  Test: 150

STEP 2: LOADING MODEL


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ Model loaded

STEP 3: TOKENIZATION
Tokenizing...


Map:   0%|          | 0/700 [00:00<?, ? examples/s]

...


✅ Training completed successfully!

STEP 8: SAVING MODEL
✓ Model saved to './finance_lora_adapter'
✓ Model zipped to 'finance_lora_adapter.zip'

STEP 9: EVALUATION



Test Results:
  eval_loss: 7.4754
  eval_word_overlap: 0.3401
  eval_exact_match: 0.0000
  eval_f1: 0.4686
  eval_runtime: 11.4391
  eval_samples_per_second: 13.1130
  eval_steps_per_second: 3.3220
  epoch: 6.0000

STEP 10: BASELINE COMPARISON
Loading baseline model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Comparing models on test samples...

Sample 1:
  Baseline F1: 0.000
  Fine-tuned F1: 0.000
  Improvement: 0.000

Sample 2:
  Baseline F1: 0.000
  Fine-tuned F1: 0.750
  Improvement: 0.750

Sample 3:
  Baseline F1: 0.000
  Fine-tuned F1: 0.400
  Improvement: 0.400

SUMMARY
Baseline F1: 0.0216
Fine-tuned F1: 0.4885
Improvement: 0.4669
P-value: 0.0000

STEP 11: GENERATING REPORT

                     MODEL COMPARISON REPORT

1. MODEL INFORMATION
   - Baseline Model: google/flan-t5-base
   - Fine-tuned Model: google/flan-t5-base + LoRA Fine-tuning
   - Evaluation Samples: 20
   - Training Samples: 700
   - Validation Samples: 150
   - Test Samples: 150
   - Test Dataset: Synthetic Financial QA Dataset (1000+ samples)
   - Epochs: 6
   - Device: cuda
   - Trainable Parameters: 3,538,944 (1.41%)

2. PERFORMANCE COMPARISON
   Metric                  | Baseline     | Fine-tuned   | Improvement
   ------------------------|--------------|--------------|-------------
   F1 Score                |